# Python 数据处理工具生态复习 — 第二阶段（2-3 个月）

这一阶段开始和数据工程真正挂钩。重点不是把每个库背熟，而是知道什么时候用什么工具、如何处理结构化数据。

| 模块 | 目标 |
|---|---|
| Pandas | 表格读取、清洗、聚合、Join |
| SQL | 查询、聚合、Join、窗口函数的基本功 |
| 文件格式 | CSV / JSON / Parquet 的适用场景 |
| API 调用 | 用 `requests` 拉取数据并处理响应 |
| 数据思维 | schema、空值、类型、重复数据、数据质量 |

---
## 1. Pandas 基础工作流

Pandas 是本地表格数据处理的最高频工具。先掌握 20% 的核心 API：

- 读取：`read_csv`, `read_json`, `read_parquet`
- 观察：`head`, `info`, `describe`, `shape`, `dtypes`
- 选择：列选择、条件过滤、`loc` / `iloc`
- 清洗：`dropna`, `fillna`, `astype`, `rename`, `drop_duplicates`
- 聚合：`groupby().agg(...)`
- 连接：`merge`

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "user_id": [101, 102, 101, 103],
    "amount": [100.0, None, 250.0, 80.0],
    "status": ["paid", "failed", "paid", "paid"],
})

orders["amount"] = orders["amount"].fillna(0).astype("float64")
paid_orders = orders[orders["status"] == "paid"]

user_stats = paid_orders.groupby("user_id").agg(
    total_amount=("amount", "sum"),
    order_count=("order_id", "count"),
).reset_index()

print(user_stats)

---
## 2. SQL 比 Python 还重要

数据工程里，大部分数据仍然在数据库、数据仓库或湖仓里。Python 常负责编排和胶水逻辑，SQL 负责计算。

### 必须扎实
- `SELECT` / `WHERE` / `ORDER BY` / `LIMIT`
- `GROUP BY` / `HAVING`
- `INNER JOIN` / `LEFT JOIN`
- `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`
- `ROW_NUMBER`, `RANK`, `LAG`, `LEAD`

建议：Python 和 SQL 不要二选一。能在 SQL 里完成的大规模聚合，通常不要先拉到 Pandas。

In [ ]:
sql = """
SELECT
  user_id,
  COUNT(*) AS order_count,
  SUM(amount) AS total_amount,
  AVG(amount) AS avg_amount
FROM orders
WHERE status = 'paid'
GROUP BY user_id
HAVING SUM(amount) > 100
ORDER BY total_amount DESC;
"""

print(sql)

---
## 3. 文件格式：CSV、JSON、Parquet

| 格式 | 优点 | 缺点 | 适用场景 |
|---|---|---|---|
| CSV | 简单、通用、人能看 | 无 schema、类型易丢、压缩差 | 小文件、人工交换 |
| JSON | 表达嵌套结构 | 体积大、解析慢 | API 响应、半结构化日志 |
| Parquet | 列式、压缩好、带 schema | 人不能直接阅读 | 数据湖、分析型批处理 |

数据工程优先级：临时交换可以 CSV/JSON，生产分析数据优先 Parquet。

In [ ]:
from pathlib import Path
import tempfile

tmp_dir = Path(tempfile.gettempdir())
csv_path = tmp_dir / "orders.csv"
json_path = tmp_dir / "orders.json"

orders.to_csv(csv_path, index=False)
orders.to_json(json_path, orient="records", lines=True, force_ascii=False)

print(pd.read_csv(csv_path).head())
print(pd.read_json(json_path, lines=True).head())

# Parquet needs pyarrow or fastparquet
# parquet_path = tmp_dir / "orders.parquet"
# orders.to_parquet(parquet_path, index=False)
# pd.read_parquet(parquet_path)

---
## 4. 基础 API 调用（requests）

数据工程常见任务：从第三方 API 拉数据，落地到文件或表。

### 必须注意
- 设置 `timeout`，不要让请求无限卡住
- 检查状态码，调用 `raise_for_status()`
- 分页、限流、重试要单独设计
- 不要把 token 写死在代码里，使用环境变量或密钥管理

In [ ]:
import requests

def fetch_json(url: str, params: dict | None = None) -> dict:
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

# Example only. Keep real API tokens out of source code.
# data = fetch_json("https://api.example.com/orders", params={"page": 1})
# df = pd.DataFrame(data["items"])


---
## 5. 数据处理时的检查清单

每次拿到新数据，先问这些问题：

1. 有多少行、多少列？`df.shape`
2. 每列类型是什么？`df.dtypes`
3. 哪些列有空值？`df.isna().sum()`
4. 主键是否重复？`df["id"].duplicated().sum()`
5. 金额、日期、状态值是否在合理范围？
6. Join 后行数有没有异常变多或变少？

练习项目：从一个 JSON API 或本地 JSON 文件读取订单数据，清洗空值，和用户维表 join，输出 Parquet。